# `DataPipeline` - Orchestrateur du flux de données kidas

**Module :** `kadi.kidas.pipeline`  
**Classe :** `DataPipeline`

---

## Introduction

`DataPipeline` est le **point d'entrée unique** du module kidas. Il orchestre l'ensemble du flux :

```
Source (CSV/Excel/JSON/NetCDF/API)
         ↓
    DataCleaner
         ↓
   DataValidator
         ↓
  DataNormalizer
         ↓
     DataCache
         ↓
   DataFrame final + rapport
```

Son **API fluide chainable** permet de définir un pipeline complexe en quelques lignes.  
L'**auto-détection de source** reconnaît l'extension du fichier ou le préfixe `https://`.

## 1. Préparation des données de test

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import os
import tempfile
import pandas as pd
import numpy as np

from kadi.kidas.pipeline import DataPipeline
from kadi.kidas.cache import DataCache

# Répertoire de cache temporaire pour les exemples
cache_dir = os.path.join(tempfile.gettempdir(), 'kidas_pipeline_demo')

# --- Création d'un fichier CSV de démonstration avec anomalies ---
df_brut = pd.DataFrame({
    'culture':      ['Maïs', 'Niébé', 'igname', 'MAÏS', None, 'sorgho'],
    'marche':       ['Dantokpa', 'Parakou', 'Bohicon', 'Dantokpa', 'Kandi', 'Natitingou'],
    'prix_xof':     [350, 500, 250, 350, 300, 280],
    'rendement_kg': [1500.0, 800.0, 2000.0, 1500.0, np.nan, 1200.0],
    'lat':          [6.366, 9.337, 7.181, 6.366, 11.133, 10.296],
    'lon':          [2.437, 2.629, 2.067, 2.437, 2.940, 1.387],
})

fichier_csv = os.path.join(tempfile.gettempdir(), 'recoltes_pipeline.csv')
df_brut.to_csv(fichier_csv, index=False)

fichier_excel = os.path.join(tempfile.gettempdir(), 'recoltes_pipeline.xlsx')
df_brut.to_excel(fichier_excel, index=False)

print(f"Fichier CSV   : {fichier_csv}")
print(f"Fichier Excel : {fichier_excel}")

Fichier CSV   : /tmp/recoltes_pipeline.csv
Fichier Excel : /tmp/recoltes_pipeline.xlsx


## 2. Utilisation rapide - `load_and_clean()`

Pour un usage simple, la fonction `load_and_clean()` du module kidas crée automatiquement un pipeline standard.

In [2]:
import kadi.kidas as kidas

# Une seule ligne pour charger et nettoyer
df_rapide, rapport_rapide = kidas.load_and_clean(fichier_csv, cache=False)

print(f"Résultat : {len(df_rapide)} lignes × {len(df_rapide.columns)} colonnes")
df_rapide

Résultat : 6 lignes × 6 colonnes


,culture,marche,prix_xof,rendement_kg,lat,lon
0,Maïs,Dantokpa,350,1500.0,6.366,2.437
1,Niébé,Parakou,500,800.0,9.337,2.629
2,igname,Bohicon,250,2000.0,7.181,2.067
3,MAÏS,Dantokpa,350,1500.0,6.366,2.437
4,NaN,Kandi,300,1400.0,11.133,2.940
5,sorgho,Natitingou,280,1200.0,10.296,1.387


## 3. `load_data()` - Configuration de la source

**Auto-détection des types :**

| Extension | Type détecté |
|---|---|
| `.csv`, `.tsv`, `.txt` | `csv` |
| `.xls`, `.xlsx`, `.xlsm` | `excel` |
| `.json` | `json` |
| `.nc`, `.nc4`, `.netcdf` | `netcdf` |
| `http://`, `https://` | `api` |

In [3]:
# Chargement depuis un fichier CSV (auto-détection)
pipeline = DataPipeline()
pipeline._cache = DataCache(cache_dir=cache_dir)
pipeline.load_data(fichier_csv)

print(f"Configuration du pipeline :")
config = pipeline.get_pipeline_config()
print(f"  Source : {config['source']['path']} (type={config['source']['type']})")
print(f"  Étapes configurées : {config['nb_etapes']}")

Configuration du pipeline :
  Source : /tmp/recoltes_pipeline.csv (type=csv)
  Étapes configurées : 0


In [4]:
# Chargement depuis un fichier Excel (auto-détection)
pipeline_excel = DataPipeline()
pipeline_excel._cache = DataCache(cache_dir=cache_dir)
pipeline_excel.load_data(fichier_excel)
config_excel = pipeline_excel.get_pipeline_config()
print(f"Type détecté pour .xlsx : {config_excel['source']['type']}")

Type détecté pour .xlsx : excel


## 4. `add_cleaning_step()` - Ajout d'étapes de nettoyage

In [5]:
pipeline = DataPipeline()
pipeline._cache = DataCache(cache_dir=cache_dir)

# Ajout de plusieurs étapes de nettoyage (chainable)
pipeline.load_data(fichier_csv)

# Étape 1 : suppression des doublons
pipeline.add_cleaning_step('remove_duplicates')

# Étape 2 : imputation des NaN par la médiane
pipeline.add_cleaning_step('handle_missing_values', strategy='median', columns=['rendement_kg'])

# Étape 3 : standardisation du texte
pipeline.add_cleaning_step('standardize_text', columns=['culture'], case='lower')

config = pipeline.get_pipeline_config()
print(f"Étapes de nettoyage configurées : {config['nb_etapes']}")
for etape in config['etapes']:
    print(f"  - {etape['nom']:30s} | params={etape['params']}")

Étapes de nettoyage configurées : 3
  - remove_duplicates              | params={}
  - handle_missing_values          | params={'strategy': 'median', 'columns': ['rendement_kg']}
  - standardize_text               | params={'columns': ['culture'], 'case': 'lower'}


## 5. `add_validation_step()` - Ajout d'une étape de validation

In [6]:
# Ajout d'une validation de schéma au pipeline
schema = {
    'culture': 'str',
    'marche':  'str',
    'prix_xof': 'float',
    'rendement_kg': 'float',
}

pipeline.add_validation_step(schema)

config = pipeline.get_pipeline_config()
print(f"Total d'étapes (avec validation) : {config['nb_etapes']}")

Total d'étapes (avec validation) : 4


## 6. `add_normalization_step()` - Ajout d'une étape de normalisation

In [7]:
# Normalisation des noms de cultures et des marchés
mappings = {
    'crops':   'culture',  # Normalise la colonne 'culture' vers FAO
    'markets': 'marche',   # Normalise la colonne 'marche' vers marchés Bénin
}

pipeline.add_normalization_step(mappings)

config = pipeline.get_pipeline_config()
print(f"Total d'étapes (avec normalisation) : {config['nb_etapes']}")

Total d'étapes (avec normalisation) : 5


## 7. `execute()` - Exécution complète du pipeline

In [8]:
# Exécution sans cache (premier run)
df_final, rapport = pipeline.execute(cache=False)

print(f"=== Résultats ===")
print(f"DataFrame final : {len(df_final)} lignes × {len(df_final.columns)} colonnes")
print(f"Lignes initiales → finales : {rapport.get('lignes_finales', 'N/A')}")
print(f"Étapes appliquées : {rapport.get('etapes_appliquees', [])}")
print(f"Cache utilisé     : {rapport.get('cache_utilise', False)}")
df_final

Validation du schéma : 1 erreur(s) détectée(s).
  - Type incorrect pour 'prix_xof' : attendu 'float', reçu 'int64'.


=== Résultats ===
DataFrame final : 6 lignes × 8 colonnes
Lignes initiales → finales : 6
Étapes appliquées : ['remove_duplicates', 'handle_missing_values', 'standardize_text', 'validate_schema', 'normalize']
Cache utilisé     : False


,culture,marche,prix_xof,rendement_kg,lat,lon,market_lat,market_lon
0,maize,Dantokpa,350,1500.0,6.366,2.437,6.366,2.437
1,cowpea,Parakou,500,800.0,9.337,2.629,9.337,2.629
2,yam,Bohicon,250,2000.0,7.181,2.067,7.181,2.067
3,maize,Dantokpa,350,1500.0,6.366,2.437,6.366,2.437
4,NaN,Kandi,300,1500.0,11.133,2.940,11.133,2.940
5,sorghum,Natitingou,280,1200.0,10.296,1.387,10.303,1.381


In [9]:
# Exécution avec cache : 2ème run → lecture directe du cache
pipeline2 = DataPipeline()
pipeline2._cache = DataCache(cache_dir=cache_dir)

df_cached, rapport_v2 = (
    pipeline2
    .load_data(fichier_csv)
    .add_cleaning_step('remove_duplicates')
    .add_cleaning_step('handle_missing_values', strategy='mean')
    .execute(cache=True)
)

print(f"Cache utilisé : {rapport_v2.get('cache_utilise', False)}")
print(f"Lignes : {len(df_cached)}")

Cache utilisé : True
Lignes : 6


## 8. Rapport complet du pipeline

In [10]:
print("=== Rapport Complet ===")

print("\nSource :")
if rapport.get('source'):
    for k, v in rapport['source'].items():
        print(f"  {k} : {v}")

print("\nNettoyage :")
if rapport.get('nettoyage'):
    nett = rapport['nettoyage']
    print(f"  Doublons supprimés : {nett.get('doublons_supprimes', 0)}")
    print(f"  NaN traités        : {nett.get('nan_traites', 0)}")

print("\nValidation :")
if rapport.get('validation'):
    val = rapport['validation']
    print(f"  Validations : {len(val.get('validations', []))}")

if rapport.get('quality_score'):
    score = rapport['quality_score']
    print(f"\nScore qualité global : {score['overall']:.2%}")

=== Rapport Complet ===

Source :
  path : /tmp/recoltes_pipeline.csv
  type : csv

Nettoyage :
  Doublons supprimés : 0
  NaN traités        : 0

Validation :
  Validations : 1

Score qualité global : 93.06%


## 9. `get_pipeline_config()` - Configuration du pipeline

In [11]:
config = pipeline.get_pipeline_config()
print("Configuration du pipeline :")
print(f"  Source       : {config['source']}")
print(f"  Nb étapes    : {config['nb_etapes']}")
for i, etape in enumerate(config['etapes'], 1):
    print(f"  Étape {i} : [{etape['type']}] {etape['nom']} → {etape['params']}")

Configuration du pipeline :
  Source       : {'path': '/tmp/recoltes_pipeline.csv', 'type': 'csv'}
  Nb étapes    : 5
  Étape 1 : [cleaning] remove_duplicates → {}
  Étape 2 : [cleaning] handle_missing_values → {'strategy': 'median', 'columns': ['rendement_kg']}
  Étape 3 : [cleaning] standardize_text → {'columns': ['culture'], 'case': 'lower'}
  Étape 4 : [validation] validate_schema → {'schema': {'culture': 'str', 'marche': 'str', 'prix_xof': 'float', 'rendement_kg': 'float'}}
  Étape 5 : [normalization] normalize → {'mappings': {'crops': 'culture', 'markets': 'marche'}}


## 10. `export_report()` - Export du rapport en JSON ou HTML

In [12]:
import json

# Export au format JSON
chemin_json = os.path.join(tempfile.gettempdir(), 'rapport_kidas.json')
succes_json = pipeline.export_report(chemin_json)
print(f"Export JSON réussi : {succes_json}")

# Affichage d'un extrait du rapport JSON
with open(chemin_json, encoding='utf-8') as f:
    rapport_json = json.load(f)

print("\nClés du rapport JSON :")
for cle in rapport_json.keys():
    print(f"  - {cle}")

Export JSON réussi : True

Clés du rapport JSON :
  - source
  - etapes_appliquees
  - nettoyage
  - validation
  - normalisation
  - cache_utilise
  - quality_score
  - lignes_finales


In [13]:
# Export au format HTML
chemin_html = os.path.join(tempfile.gettempdir(), 'rapport_kidas.html')
succes_html = pipeline.export_report(chemin_html)
print(f"Export HTML réussi  : {succes_html}")
print(f"Rapport HTML disponible : {chemin_html}")

Export HTML réussi  : True
Rapport HTML disponible : /tmp/rapport_kidas.html


## 11. Pipelines avancés - Exemples complets

In [14]:
# === Pipeline complet pour les prix de marchés ===
pipeline_marches = DataPipeline()
pipeline_marches._cache = DataCache(cache_dir=cache_dir)

df_marches, rapport_marches = (
    pipeline_marches
    .load_data(fichier_csv)
    .add_cleaning_step('remove_duplicates')
    .add_cleaning_step('handle_missing_values', strategy='median')
    .add_cleaning_step('standardize_text', columns=['culture'], case='lower')
    .add_validation_step({'culture': 'str', 'prix_xof': 'float'})
    .add_normalization_step({'crops': 'culture', 'markets': 'marche'})
    .execute(cache=False)
)

print(f"Pipeline marchés : {len(df_marches)} lignes")
df_marches[['culture', 'marche', 'prix_xof', 'rendement_kg']].head()

Validation du schéma : 1 erreur(s) détectée(s).
  - Type incorrect pour 'prix_xof' : attendu 'float', reçu 'int64'.


Pipeline marchés : 6 lignes


,culture,marche,prix_xof,rendement_kg
0,maize,Dantokpa,350,1500.0
1,cowpea,Parakou,500,800.0
2,yam,Bohicon,250,2000.0
3,maize,Dantokpa,350,1500.0
4,NaN,Kandi,300,1500.0


In [15]:
# === Utilisation avec DataSource directe ===
from kadi.kidas.sources.csv_source import CSVDataSource

source_directe = CSVDataSource(fichier_csv, encoding='utf-8')

pipeline_direct = DataPipeline()
pipeline_direct._cache = DataCache(cache_dir=cache_dir)

df_direct, _ = (
    pipeline_direct
    .load_data(source_directe)  # DataSource directe (pas de chemin)
    .add_cleaning_step('remove_duplicates')
    .execute(cache=False)
)

print(f"Via DataSource directe : {len(df_direct)} lignes")

Via DataSource directe : 6 lignes


## 12. `_detecter_type_source()` - Détection du type de source

In [16]:
# Méthode statique : peut être utilisée indépendamment
exemples = [
    'recoltes.csv',
    'prix_marches.xlsx',
    'donnees_fao.json',
    'chirps_benin.nc',
    'https://api.open-meteo.com/forecast',
]

print("Détection automatique du type de source :")
for chemin in exemples:
    try:
        type_detecte = DataPipeline._detecter_type_source(chemin)
        print(f"  {chemin:40s} → {type_detecte}")
    except Exception as e:
        print(f"  {chemin:40s} → ERREUR : {e}")

Détection automatique du type de source :
  recoltes.csv                             → csv
  prix_marches.xlsx                        → excel
  donnees_fao.json                         → json
  chirps_benin.nc                          → netcdf
  https://api.open-meteo.com/forecast      → api


## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `load_data(source, **kwargs)` | Configure la source (auto-détection du type) |
| `add_cleaning_step(name, **params)` | Ajoute une étape DataCleaner |
| `add_validation_step(schema)` | Ajoute une validation DataValidator |
| `add_normalization_step(mappings)` | Ajoute une normalisation DataNormalizer |
| `execute(cache)` | Lance le pipeline et retourne (df, rapport) |
| `get_pipeline_config()` | Retourne la configuration complète |
| `export_report(filepath)` | Exporte le rapport en JSON ou HTML |
| `_detecter_type_source(path)` | Détecte le type depuis l'extension ou URL |

---

> **Conseil :** Pour les scripts de production, utilisez `cache=True` et définissez le répertoire cache dans `~/.kadi/kidas_cache/`. Cela évite de recharger des données déjà traitées lors des relances.